# Elastic Net Phase 2: macro-gain isolation (no-macro EN, VIX ablation, HAR lags, QLIKE)

Duplicated from `elastic_net_midterm1.ipynb` (kept as-is for midterm reference) and
extended to answer the professor's Phase 2 feedback: EN alone can't show *where* any
gain over GARCH comes from (the macro data, or just persistence-modelling at monthly
frequency). This notebook runs **six** Elastic Net variants through the same
walk-forward engine -- two persistence specs (single-lag, HAR-style 1/3/12-month),
each crossed with three macro-information levels (full macro, no VIX, no macro) --
so the macro-gain and VIX-ablation questions can be answered under either spec, and
compared against the matching `ols_1m`/`ols_1_3_12m` baselines in `ols_phase2.ipynb`:

- **`en_full_macro`** / **`en_full_macro_har`** — 11 macro vars + persistence terms.
- **`en_no_vix`** / **`en_no_vix_har`** — same, minus `VIXCLS` (implied vol is the
  market's own forecast, not a Chen-Roll-Ross macro channel — isolates "implied vol
  helps" from "macro helps").
- **`en_no_macro`** / **`en_no_macro_har`** — no macro at all, persistence terms only.
  The benchmark the macro-gain claim is actually measured against, matching
  `ols_1m`/`ols_1_3_12m` exactly.

All six are Elastic Net (same `ElasticNetCV` hyperparameter search + regularized
fit), just with different feature sets — isolates the macro-information question
without also changing estimation method between variants.

All six keep the same expanding-window walk-forward design as midterm1, but now
refit **every month** (not yearly) — midterm1 used a yearly freeze only because of
that deadline's time constraint; for Phase 2, GARCH/GARCH-MIDAS are also refitting
monthly, so matching that cadence keeps the comparison fair. Outputs include an
out-of-fold training-window residual variance (`sigma2`) and a log-normal corrected
level forecast (`fc_level`), so MSE (log scale) and QLIKE (level scale) can both be
reported.

# What this notebook does

Forecasts next-month log realized variance (`log RV²`) for 5 cyclical sector ETFs
(XLI, XLB, XLE, XLY, XLF), and runs **six Elastic Net variants** through the same
walk-forward engine so they're directly comparable: `en_full_macro`, `en_no_vix`,
`en_no_macro` (single-lag) and their `_har` counterparts (1/3/12-month lags). Produces
the common-basis outputs (log forecast + sigma2 + log-normal corrected level forecast)
and an MSE/QLIKE comparison table across all six — the actual answer to "does macro
information help, and is it just VIX?", under both a single-lag and a HAR persistence
specification.

**Changes made in this version vs. midterm1** (see Phase 2 feedback + investigation notes):
1. **RV data source**: reads the shared `dataset/price/clean/realized_variance.csv`
   (built from `daily_returns_panel_ver2.csv` -- XLRE correctly spliced into XLF --
   using excess returns `xret_*`, with January 2003 excluded for the same
   single-trading-day-month reason this notebook used to patch locally) instead of
   recomputing RV² here. Guarantees byte-identical RV numbers with `ols_phase2.ipynb`.
2. **Macro set**: locked to the **11 vars named in the proposal** (`T10Y2Y, DGS3MO,
   BAA10Y, CPIAUCSL, T5YIE, T5YIE_MINUS_T10YIE, VIXCLS, INDPRO, UNRATE, UMCSENT,
   DCOILWTICO`) — two more variables were added to `final_macro_data.csv` after the
   proposal was reviewed; this notebook deliberately stays on what was submitted.
3. **Refit cadence**: monthly, not yearly. Midterm1 froze hyperparameters+coefficients
   for 12 months at a time purely because of that deadline's time constraint. For
   Phase 2, GARCH/GARCH-MIDAS are refitting monthly too, so EN now matches that.
4. **HAR-style variants**: added `en_*_har` models (1/3/12-month lags) alongside the
   original single-lag models, matching `ols_phase2.ipynb`'s `ols_1m`/`ols_1_3_12m`
   split -- lets the macro-gain/VIX questions be answered under either persistence spec.

**Walk-forward mechanics**: burn-in 60 months (counted per-model from that model's own
first valid row -- the HAR variants' sample starts later), expanding window, refit
(hyperparameters + coefficients together) every month. (Midterm1's rolling-window side
experiment was dropped here — expanding only.)

**Not in this notebook** (flagged for the team, not built here): portfolio weights,
Diebold-Mariano tests against GARCH/XGBoost (need their forecast files), pooled EN
with ETF dummies (depends on the XGBoost teammate's pooling decision). The tidy output
format here is shaped to make a later merge with those easy, once they exist.

In [1]:
import os
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import ElasticNetCV, ElasticNet, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.exceptions import ConvergenceWarning

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = "../.."
CYCLICAL_TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]

MACRO_PATH = os.path.join(PROJECT_ROOT, "dataset", "macro_variables", "final_clean", "final_macro_data.csv")
# shared, precomputed RV panel (dataset/price/calculate_realized_variance.py) -- same
# daily_returns_panel_ver2.csv + xret_ + Jan-2003-exclusion construction as this notebook
# used to do locally, now sourced from one place so EN and the OLS/HAR baseline notebook
# (scripts/ols/ols_phase2.ipynb) are guaranteed byte-identical on the RV side.
RV_PATH = os.path.join(PROJECT_ROOT, "dataset", "price", "clean", "realized_variance.csv")
FORECASTS_DIR = os.path.join(PROJECT_ROOT, "forecasts")  # top-level, not nested under dataset/

BURN_IN = 60       # months of history required before the first fit/forecast
REFIT_EVERY = 1    # months between hyperparameter search + coefficient refits -- monthly
                    # for Phase 2 (GARCH is also refitting monthly now, so cadence stays
                    # matched; midterm1 used 12 only because of the time constraint then)
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

## Data

Macro predictors from `final_macro_data.csv`, restricted to the 11 proposal-specified
variables (already vintage/publication-lag corrected, transformed level-vs-log-diff
per channel upstream). RV side from the shared `realized_variance.csv` (built by
`dataset/price/calculate_realized_variance.py` from excess returns, same
Jan-2003-exclusion fix this notebook used to apply locally) -- gives `log_rv_1m`
(current month's log RV²) plus `log_rv_avg_3m`/`log_rv_avg_12m` (trailing 3-/12-month
averages of `log_rv_1m`, for the HAR-style variants). `X.loc[t]` (known at month t)
pairs with `y_target.loc[t] = log_rv_1m` realized in month t+1.

In [2]:
# Explicit allowlist (not "all columns minus NBUSBIS" -- that was dead code, since NBUSBIS
# is already gone upstream, and would have silently pulled in DEXUSEU/USEPUINDXD, which
# were added to final_macro_data.csv *after* the proposal's 11-variable set was reviewed).
FINAL_MACRO_VARS = [
    "T10Y2Y", "DGS3MO", "BAA10Y", "CPIAUCSL", "T5YIE", "T5YIE_MINUS_T10YIE",
    "VIXCLS", "INDPRO", "UNRATE", "UMCSENT", "DCOILWTICO",
]

macro = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)
macro = macro[FINAL_MACRO_VARS].copy()
print(f"{len(FINAL_MACRO_VARS)} macro variables (proposal set, vintage/publication-lag corrected):", FINAL_MACRO_VARS)
print(f"macro range: {macro.index.min().date()} to {macro.index.max().date()}\n")

rv = pd.read_csv(RV_PATH, parse_dates=["date"])
RV_FEATURE_COLS = ["log_rv_1m", "log_rv_avg_3m", "log_rv_avg_12m"]
print(f"realized_variance.csv: {rv['date'].min().date()} to {rv['date'].max().date()}, "
      f"etfs: {sorted(rv['etf'].unique())}\n")

# One combined per-ticker frame: macro + the 3 RV feature columns + next-month target.
# NOT dropna'd here -- log_rv_avg_3m/12m are legitimately NaN for the first 2/11 months
# of each ticker's history (not enough trailing data yet), and different model variants
# need different numbers of these columns, so each model drops its own NaNs later (same
# convention as ols_phase2.ipynb's fit_forecasts: dropna(subset=features + ["target"])
# per model, not one shared cutoff for everyone).
aligned_data = {}
for ticker in CYCLICAL_TICKERS:
    rv_ticker = (
        rv.loc[rv["etf"] == ticker, ["date"] + RV_FEATURE_COLS]
        .set_index("date")
        .sort_index()
    )
    y_target = rv_ticker["log_rv_1m"].shift(-1).rename("y_target")
    df = macro.join(rv_ticker, how="inner").join(y_target, how="inner")
    aligned_data[ticker] = df
    n_valid_1m = df["log_rv_1m"].notna().sum()
    n_valid_har = df[RV_FEATURE_COLS].notna().all(axis=1).sum()
    print(f"{ticker}: {len(df)} rows total ({df.index.min().date()} to {df.index.max().date()}), "
          f"{n_valid_1m} with log_rv_1m, {n_valid_har} with all 3 RV features (HAR-ready)")

ALL_MACRO_AND_1M = FINAL_MACRO_VARS + ["log_rv_1m"]
print()
print(f"{len(ALL_MACRO_AND_1M)} features (full EN, single-lag model):", ALL_MACRO_AND_1M)

11 macro variables (proposal set, vintage/publication-lag corrected): ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO']
macro range: 2003-01-31 to 2026-07-31

realized_variance.csv: 2003-02-28 to 2025-12-31, etfs: ['XLB', 'XLE', 'XLF', 'XLI', 'XLY']

XLI: 275 rows total (2003-02-28 to 2025-12-31), 275 with log_rv_1m, 264 with all 3 RV features (HAR-ready)
XLB: 275 rows total (2003-02-28 to 2025-12-31), 275 with log_rv_1m, 264 with all 3 RV features (HAR-ready)
XLE: 275 rows total (2003-02-28 to 2025-12-31), 275 with log_rv_1m, 264 with all 3 RV features (HAR-ready)
XLY: 275 rows total (2003-02-28 to 2025-12-31), 275 with log_rv_1m, 264 with all 3 RV features (HAR-ready)
XLF: 275 rows total (2003-02-28 to 2025-12-31), 275 with log_rv_1m, 264 with all 3 RV features (HAR-ready)

12 features (full EN, single-lag model): ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO',

## Six Elastic Net variants, one shared engine

Two persistence specifications, each crossed with the same three macro-information
levels, so the macro-gain and VIX-ablation questions can be answered under *either*
specification:

**Single-lag** (matches `ols_1m`, `own_log_rv_sq_lag0` from earlier in the project):
- **`en_full_macro`** — 11 macro vars + `log_rv_1m`.
- **`en_no_vix`** — 10 macro vars (no `VIXCLS`) + `log_rv_1m`.
- **`en_no_macro`** — `log_rv_1m` only. Benchmark for the single-lag comparison.

**HAR-style** (matches `ols_1_3_12m` -- adds trailing 3-/12-month averages, per the
professor's "if you want a HAR-style version, EN must include the same terms"):
- **`en_full_macro_har`** — 11 macro vars + `log_rv_1m` + `log_rv_avg_3m` + `log_rv_avg_12m`.
- **`en_no_vix_har`** — 10 macro vars (no `VIXCLS`) + the same 3 RV terms.
- **`en_no_macro_har`** — the 3 RV terms only. Benchmark for the HAR comparison --
  matches `ols_1_3_12m` exactly (same features, same sample).

All six are Elastic Net (same `ElasticNetCV` search), only the feature set differs --
keeps the macro-information question isolated from any estimation-method difference.
The HAR variants naturally start later and have fewer forecasts than the single-lag
ones (12 months needed before `log_rv_avg_12m` exists), same as `ols_1_3_12m` vs.
`ols_1m` -- each model's own `dropna()` on its feature set determines its sample,
not a shared cutoff.

In [3]:
NO_VIX_MACRO = [c for c in FINAL_MACRO_VARS if c != "VIXCLS"]

FEATURE_SETS = {
    "en_full_macro": FINAL_MACRO_VARS + ["log_rv_1m"],
    "en_no_vix": NO_VIX_MACRO + ["log_rv_1m"],
    "en_no_macro": ["log_rv_1m"],
    "en_full_macro_har": FINAL_MACRO_VARS + RV_FEATURE_COLS,
    "en_no_vix_har": NO_VIX_MACRO + RV_FEATURE_COLS,
    "en_no_macro_har": list(RV_FEATURE_COLS),
}
# all six are Elastic Net -- same ElasticNetCV hyperparameter search, only the
# feature set differs between them (see markdown above)
MODEL_KINDS = {name: "elasticnet" for name in FEATURE_SETS}

for name, cols in FEATURE_SETS.items():
    print(f"{name:18s}: {len(cols)} features -> {cols}")

en_full_macro     : 12 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'log_rv_1m']
en_no_vix         : 11 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'log_rv_1m']
en_no_macro       : 1 features -> ['log_rv_1m']
en_full_macro_har : 14 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'VIXCLS', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'log_rv_1m', 'log_rv_avg_3m', 'log_rv_avg_12m']
en_no_vix_har     : 13 features -> ['T10Y2Y', 'DGS3MO', 'BAA10Y', 'CPIAUCSL', 'T5YIE', 'T5YIE_MINUS_T10YIE', 'INDPRO', 'UNRATE', 'UMCSENT', 'DCOILWTICO', 'log_rv_1m', 'log_rv_avg_3m', 'log_rv_avg_12m']
en_no_macro_har   : 3 features -> ['log_rv_1m', 'log_rv_avg_3m', 'log_rv_avg_12m']


## Walk-forward engine — shared by all 6 models, expanding window, monthly refit

Renamed from `walk_forward_yearly` -- that name was a holdover from midterm1's
yearly-freeze design; since Phase 2 refits every month (`REFIT_EVERY = 1`), "yearly"
no longer describes it, so it's just `walk_forward` now.

- **Per-model sample**: each model gets `model_df = aligned_data[ticker][feature_cols
  + ["y_target"]].dropna()` before the loop even starts -- same convention as
  `ols_phase2.ipynb`'s `fit_forecasts()`. The single-lag models' sample is unaffected
  (only `log_rv_1m` + macro, always present); the HAR models' sample starts ~11 months
  later since `log_rv_avg_12m` needs a full trailing year. `burn_in` is counted from
  each model's own first valid row, not a shared position.
- At each `REFIT_EVERY`-month cycle: `ElasticNetCV` (5-fold `TimeSeriesSplit`) picks
  `alpha`/`l1_ratio` on the training window (`X[0:i]`, grows every cycle), then a plain
  `ElasticNet` is fit with those. Frozen until the next cycle.
- **Out-of-fold `sigma2`**: manually loop the *same* `TimeSeriesSplit(5)` splits used
  for hyperparameter selection to get genuinely held-out predictions over the training
  window -- no extra look-ahead, avoids the in-sample-understates-error problem of a
  regularized fit.
- **Log-normal corrected level forecast**: `fc_level = exp(fc_log + 0.5*sigma2)`.

In [4]:
def walk_forward(model_df, feature_cols, model_kind, window=None,
                  burn_in=BURN_IN, refit_every=REFIT_EVERY, l1_ratios=L1_RATIOS):
    """Shared walk-forward engine for all 6 EN variants, expanding or rolling.

    model_df: already filtered to this model's own valid rows (dropna on feature_cols
    + "y_target") -- see run_all_models, same convention as ols_phase2.ipynb's
    fit_forecasts(). window=None -> expanding (train on X[:i]); window=60 -> rolling
    (train on X[i-window:i], unused in this notebook but kept for generality). At each
    refit_every-month cycle: pick hyperparameters (elasticnet) or just fit (ols --
    supported for generality, unused here since all 6 variants are Elastic Net), refit
    the final model, scaler frozen with it, and compute out-of-fold sigma2 by manually
    looping the same TimeSeriesSplit(5) used for hyperparameter selection -- all frozen
    until the next cycle. Returns one row per forecast month: log forecast, sigma2,
    log-normal corrected level forecast, plus actuals and hyperparameters used.
    """
    X = model_df[feature_cols].to_numpy()
    y = model_df["y_target"].to_numpy()
    dates = model_df.index
    n = len(model_df)
    start = burn_in if window is None else window
    rows = []
    pipe = None
    sigma2 = np.nan
    alpha_ = l1_ratio_ = np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        for i in range(start, n):
            train_start = 0 if window is None else i - window
            refit_this_month = (i - start) % refit_every == 0
            if refit_this_month:
                X_train, y_train = X[train_start:i], y[train_start:i]
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)

                if model_kind == "elasticnet":
                    cv_search = ElasticNetCV(
                        l1_ratio=l1_ratios, cv=TimeSeriesSplit(n_splits=5),
                        max_iter=10000, random_state=0,
                    )
                    cv_search.fit(X_train_scaled, y_train)
                    alpha_, l1_ratio_ = cv_search.alpha_, cv_search.l1_ratio_
                    model = ElasticNet(alpha=alpha_, l1_ratio=l1_ratio_, max_iter=10000, random_state=0)
                else:  # "ols" -- no hyperparameters to search
                    alpha_ = l1_ratio_ = np.nan
                    model = LinearRegression()
                model.fit(X_train_scaled, y_train)

                # Out-of-fold residuals on the SAME training window + CV splits used for
                # hyperparameter selection -- no extra look-ahead, avoids the
                # in-sample-understates-error problem of a regularized fit. TimeSeriesSplit
                # never tests its very first (smallest) training chunk, so this isn't a full
                # partition -- loop the splits manually instead of sklearn's
                # cross_val_predict (which requires one), and only use whichever rows
                # actually got tested.
                oof_pred = np.full(len(y_train), np.nan)
                for tr_idx, te_idx in TimeSeriesSplit(n_splits=5).split(X_train_scaled):
                    fold_model = clone(model)
                    fold_model.fit(X_train_scaled[tr_idx], y_train[tr_idx])
                    oof_pred[te_idx] = fold_model.predict(X_train_scaled[te_idx])
                tested = ~np.isnan(oof_pred)
                sigma2 = float(np.var(y_train[tested] - oof_pred[tested]))

                # frozen scaler + frozen model, wrapped so .predict() applies both in order
                pipe = Pipeline([("scaler", scaler), ("model", model)])

            pred_log = pipe.predict(X[i:i + 1])[0]
            pred_level = np.exp(pred_log + 0.5 * sigma2)  # log-normal correction
            rows.append({
                "date": dates[i],
                "target_date": dates[i] + pd.offsets.MonthEnd(1),
                "actual_logRV": y[i],
                "actual_RV": np.exp(y[i]),
                "fc_log": pred_log,
                "sigma2": sigma2,
                "fc_level": pred_level,
                "alpha": alpha_,
                "l1_ratio": l1_ratio_,
                "refit_this_month": refit_this_month,
                "train_months": i - train_start,
            })
    return pd.DataFrame(rows).set_index("date")

In [5]:
def run_all_models(window, label):
    """Runs all 6 FEATURE_SETS variants for all 5 tickers at the given window setting
    (None=expanding, 60=rolling). Each model's own feature_cols determine its sample
    (dropna per model, not a shared cutoff -- the HAR variants naturally end up with
    fewer, later-starting forecasts than the single-lag ones). Returns (wide_by_model, tidy_df):
      - wide_by_model[model][ticker] = per-ticker forecast frame (for accuracy_table)
      - tidy_df = one long table, all models/tickers/months stacked -- the common-basis
        output meant to be merged later with GARCH/GARCH-MIDAS/XGB forecasts.
    """
    wide_by_model = {}
    tidy_rows = []
    t0 = time.time()
    for model_name, feature_cols in FEATURE_SETS.items():
        wide_by_model[model_name] = {}
        for ticker in CYCLICAL_TICKERS:
            model_df = aligned_data[ticker][feature_cols + ["y_target"]].dropna()
            df = walk_forward(model_df, feature_cols, MODEL_KINDS[model_name], window=window)
            wide_by_model[model_name][ticker] = df
            tidy = df.reset_index()[
                ["date", "target_date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]
            ].copy()
            tidy.insert(0, "etf", ticker)
            tidy.insert(0, "model", model_name)
            tidy_rows.append(tidy)
        n_xli = len(wide_by_model[model_name]["XLI"])
        print(f"{model_name:18s}: {n_xli} forecasts/ticker, {time.time() - t0:.1f}s elapsed")
    tidy_df = pd.concat(tidy_rows, ignore_index=True)

    os.makedirs(FORECASTS_DIR, exist_ok=True)
    out_path = os.path.join(FORECASTS_DIR, f"en_forecasts_{label}_tidy.csv")
    tidy_df.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(tidy_df)} rows total -- sample length differs per model, "
          f"see per-model forecast counts above)")
    return wide_by_model, tidy_df


forecasts_expanding_by_model, tidy_expanding_df = run_all_models(window=None, label="expanding")

en_full_macro     : 214 forecasts/ticker, 268.9s elapsed
en_no_vix         : 214 forecasts/ticker, 545.6s elapsed
en_no_macro       : 214 forecasts/ticker, 778.4s elapsed
en_full_macro_har : 203 forecasts/ticker, 1049.8s elapsed
en_no_vix_har     : 203 forecasts/ticker, 1308.9s elapsed
en_no_macro_har   : 203 forecasts/ticker, 1534.0s elapsed
Saved ../../forecasts/en_forecasts_expanding_tidy.csv (6255 rows total -- sample length differs per model, see per-model forecast counts above)


## Accuracy (log scale) — diagnostic, per model

Same MSE/RMSE/MAE + R²_OOS (vs. leak-free expanding historical mean, and vs.
persistence) as before, now looped over all 6 models. Since the 3 HAR models have a
different (later-starting, shorter) sample than the 3 single-lag models, the
historical-mean benchmark is computed from each ticker's full `y_target` series and
matched to each model's forecast dates -- not a fixed row position -- so it stays
correct regardless of which model's sample is being scored. Useful as a mid-notebook
sanity check; the table that actually answers the Phase 2 question (MSE + QLIKE) is
further down.

In [6]:
def historical_mean_benchmark_full(y_target_full):
    """Leak-free expanding mean of the FULL (un-filtered) y_target series for a
    ticker -- mean of every target value strictly before each date. Computed once per
    ticker from the full series (same regardless of model, since y_target doesn't
    depend on feature_cols), then reindexed to whichever dates a given model's
    forecasts actually cover."""
    return y_target_full.expanding().mean().shift(1)


def accuracy_table(forecasts, y_target_full_by_ticker, label):
    """forecasts: dict ticker -> walk_forward() output (actual_logRV/fc_log cols).
    y_target_full_by_ticker: dict ticker -> full y_target series (aligned_data[ticker]["y_target"])."""
    rows, persistence_rows = [], []
    for ticker in CYCLICAL_TICKERS:
        df = forecasts[ticker]
        actual = df["actual_logRV"]
        pred = df["fc_log"]

        benchmark_full = historical_mean_benchmark_full(y_target_full_by_ticker[ticker])
        benchmark = benchmark_full.reindex(df.index)
        resid = actual - pred
        mse = (resid ** 2).mean()
        r2_oos = 1 - (resid ** 2).sum() / ((actual - benchmark) ** 2).sum()
        rows.append({
            "ticker": ticker, "n_forecasts": len(df),
            "MSE": mse, "RMSE": np.sqrt(mse), "MAE": resid.abs().mean(), "R2_OOS": r2_oos,
        })

        persistence_pred = actual.shift(1)
        valid = persistence_pred.notna()
        resid_model = (actual - pred)[valid]
        resid_naive = (actual - persistence_pred)[valid]
        r2_vs_persistence = 1 - (resid_model ** 2).sum() / (resid_naive ** 2).sum()
        persistence_rows.append({
            "ticker": ticker,
            "MSE_model": (resid_model ** 2).mean(),
            "MSE_persistence": (resid_naive ** 2).mean(),
            "R2_OOS_vs_persistence": r2_vs_persistence,
        })

    acc_df = pd.DataFrame(rows).set_index("ticker")
    pers_df = pd.DataFrame(persistence_rows).set_index("ticker")
    print(f"=== {label}: accuracy ===")
    print(acc_df)
    print()
    print(f"=== {label}: vs persistence baseline ===")
    print(pers_df)
    return acc_df, pers_df


y_target_full_by_ticker = {t: aligned_data[t]["y_target"] for t in CYCLICAL_TICKERS}

accuracy_expanding = {}
for model_name in FEATURE_SETS:
    accuracy_expanding[model_name] = accuracy_table(
        forecasts_expanding_by_model[model_name], y_target_full_by_ticker,
        f"Expanding window, {model_name}"
    )

=== Expanding window, en_full_macro: accuracy ===
        n_forecasts       MSE      RMSE       MAE    R2_OOS
ticker                                                     
XLI             214  0.455388  0.674824  0.482421  0.499264
XLB             214  0.420332  0.648330  0.478592  0.498115
XLE             214  0.469333  0.685079  0.493816  0.500472
XLY             214  0.491828  0.701305  0.536920  0.532074
XLF             214  0.613908  0.783523  0.592812  0.572537

=== Expanding window, en_full_macro: vs persistence baseline ===
        MSE_model  MSE_persistence  R2_OOS_vs_persistence
ticker                                                   
XLI      0.452702         0.629000               0.280282
XLB      0.417680         0.536835               0.221959
XLE      0.466707         0.513143               0.090492
XLY      0.493639         0.578056               0.146036
XLF      0.611645         0.755288               0.190183
=== Expanding window, en_no_vix: accuracy ===
        n_fo

## MSE (log RV) + QLIKE (RV level) — the table that answers the Phase 2 question

Two losses, each on the scale they're meaningful for:
- **`MSE_logRV`** = `mean((actual_logRV - fc_log)^2)` — same log-scale error used
  throughout this notebook so far.
- **`QLIKE`** = `RV/RV_hat - log(RV/RV_hat) - 1`, computed on the **level** scale using
  the log-normal corrected `fc_level` (not naive `exp(fc_log)`). QLIKE is robust to
  noise in the RV proxy (MSE on log RV isn't), and it's the loss that matches what the
  portfolio step actually uses (levels, not logs) — per the Phase 2 feedback, "two
  losses are enough."

**How to read the table:** within each persistence spec (single-lag vs. HAR), compare
`..._no_macro` against `..._full_macro` row-by-row — if `_full_macro` is clearly
better, that's evidence the 11 macro variables add real value beyond the persistence
terms alone. Compare `_full_macro` against `_no_vix` the same way for the VIX-specific
question: if the edge disappears without VIX, the finding is "implied vol helps"
rather than "macro information helps". **Don't compare across specs directly** (e.g.
`en_full_macro` vs. `en_full_macro_har`) without checking they cover the same dates --
the HAR models have a shorter, later-starting sample. `pooled` columns combine all 5
tickers' residuals into one score, per model.

In [7]:
def qlike(actual_rv, fc_level):
    ratio = actual_rv / fc_level
    return ratio - np.log(ratio) - 1


def mse_qlike_table(tidy_df, label):
    """Returns (mse_wide, qlike_wide): rows = model, columns = ticker (+ pooled) --
    read down a column to compare models on the same ticker."""
    tidy_df = tidy_df.copy()
    tidy_df["sq_err_log"] = (tidy_df["actual_logRV"] - tidy_df["fc_log"]) ** 2
    tidy_df["qlike"] = qlike(tidy_df["actual_RV"], tidy_df["fc_level"])

    per_ticker = (
        tidy_df.groupby(["model", "etf"])
        .agg(MSE_logRV=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
        .reset_index()
    )
    pooled = (
        tidy_df.groupby("model")
        .agg(MSE_logRV=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
        .reset_index()
    )
    pooled["etf"] = "pooled"
    long = pd.concat([per_ticker, pooled], ignore_index=True)

    model_order = list(FEATURE_SETS)
    ticker_order = CYCLICAL_TICKERS + ["pooled"]
    mse_wide = long.pivot(index="model", columns="etf", values="MSE_logRV").reindex(index=model_order, columns=ticker_order)
    qlike_wide = long.pivot(index="model", columns="etf", values="QLIKE").reindex(index=model_order, columns=ticker_order)

    print(f"=== {label}: MSE (log RV), model x ticker ===")
    print(mse_wide)
    print()
    print(f"=== {label}: QLIKE (RV level), model x ticker ===")
    print(qlike_wide)
    return mse_wide, qlike_wide


mse_expanding, qlike_expanding = mse_qlike_table(tidy_expanding_df, "Expanding window")

=== Expanding window: MSE (log RV), model x ticker ===
etf                     XLI       XLB       XLE       XLY       XLF    pooled
model                                                                        
en_full_macro      0.455388  0.420332  0.469333  0.491828  0.613908  0.490158
en_no_vix          0.528692  0.479018  0.481542  0.525842  0.670305  0.537080
en_no_macro        0.529142  0.462059  0.452327  0.504936  0.665312  0.522755
en_full_macro_har  0.401806  0.382419  0.426828  0.484532  0.550472  0.449212
en_no_vix_har      0.492933  0.433568  0.441634  0.530786  0.608051  0.501394
en_no_macro_har    0.494616  0.432942  0.423410  0.506687  0.611924  0.493916

=== Expanding window: QLIKE (RV level), model x ticker ===
etf                     XLI       XLB       XLE       XLY       XLF    pooled
model                                                                        
en_full_macro      0.300235  0.279468  0.357333  0.293290  0.393131  0.324691
en_no_vix          0.394995